# Pré-processamento e Engenharia de Features

Este notebook implementa o pipeline de pré-processamento com as seguintes melhorias:

1. **Correção de Data Leakage**: Scaler é fitado apenas nos dados de treino
2. **Janelas Configuráveis**: Experimentos com W=3, W=7, W=15
3. **Modularização**: Uso de funções do módulo `src/features.py`

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.preprocessing import StandardScaler
import joblib

sys.path.append(os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'src'))
from features import preparar_features, criar_features_multi_horizonte

## 1. Carregamento dos Dados

In [ ]:
df = pd.read_parquet("../data/processed/vendas_supermercado.parquet", engine="pyarrow")
print("Shape original:", df.shape)

In [ ]:
df.info()

## 2. Seleção do Top SKU

In [ ]:
produto_top = (
    df.groupby("Cod_Produto")["Quantidade"]
      .sum()
      .idxmax()
)

print("Produto mais vendido:", produto_top)

In [ ]:
df_top = df[df["Cod_Produto"] == produto_top].copy()

In [ ]:
df_top["Data"] = pd.to_datetime(df_top["Data"])

In [ ]:
df_daily = (
    df_top.groupby("Data")["Quantidade"]
    .sum()
    .reset_index()
    .sort_values("Data")
)

df_daily

In [ ]:
df_daily.describe()

## 3. Engenharia de Features com Janelas Configuráveis

### Experimento 1: Configuração Padrão (lag=7, W=7)

In [ ]:
df_features = preparar_features(df_daily, max_lag=7, janela_rolling=7)
print("Shape após feature engineering:", df_features.shape)
df_features.head()

## 4. Divisão Temporal (ANTES do Scaling)

**IMPORTANTE**: Esta é a correção do data leakage. Dividimos os dados ANTES de aplicar o scaler.

In [ ]:
split_index = int(len(df_features) * 0.8)

df_train = df_features.iloc[:split_index].copy()
df_test = df_features.iloc[split_index:].copy()

print(f"Treino: {len(df_train)} registros")
print(f"Teste: {len(df_test)} registros")

## 5. Padronização (Fit apenas no Treino)

**Correção Crítica**: O scaler é fitado APENAS nos dados de treino, evitando vazamento de informação futura.

In [ ]:
colunas_numericas = df_train.columns.drop("Data")

scaler = StandardScaler()

df_train[colunas_numericas] = scaler.fit_transform(df_train[colunas_numericas])
df_test[colunas_numericas] = scaler.transform(df_test[colunas_numericas])

print("Scaler fitado apenas nos dados de treino!")

In [ ]:
df_train.head()

## 6. Salvamento dos Dados Processados

In [ ]:
df_train.to_parquet("../data/processed/train_features.parquet", engine="pyarrow")
df_test.to_parquet("../data/processed/test_features.parquet", engine="pyarrow")
print("Dados de treino e teste salvos!")

In [ ]:
joblib.dump(scaler, '../models/scaler_quantidade.pkl')
print("Scaler salvo em ../models/scaler_quantidade.pkl")

## 7. Experimentos com Diferentes Janelas

### Matriz de Configurações

In [ ]:
configuracoes = [
    (3, 3),    # Lags curtos, janela curta
    (7, 7),    # Configuração padrão
    (15, 15),  # Lags longos, janela longa
    (7, 3),    # Lags padrão, janela curta
    (7, 15),   # Lags padrão, janela longa
    (3, 7)     # Lags curtos, janela padrão
]

from features import preparar_features_multiplas_janelas
resultados_configs = preparar_features_multiplas_janelas(df_daily, configuracoes)

for nome, df_config in resultados_configs.items():
    print(f"{nome}: {df_config.shape}")

### Salvar Todas as Configurações

In [ ]:
for nome, df_config in resultados_configs.items():
    split_idx = int(len(df_config) * 0.8)
    df_t = df_config.iloc[:split_idx].copy()
    df_te = df_config.iloc[split_idx:].copy()
    
    cols = df_t.columns.drop("Data")
    scaler_temp = StandardScaler()
    df_t[cols] = scaler_temp.fit_transform(df_t[cols])
    df_te[cols] = scaler_temp.transform(df_te[cols])
    
    df_t.to_parquet(f"../data/processed/train_{nome}.parquet", engine="pyarrow")
    df_te.to_parquet(f"../data/processed/test_{nome}.parquet", engine="pyarrow")

print("Todas as configurações salvas!")

## 8. Preparação para MIMO (Multi-Input Multi-Output)

Criação de targets para múltiplos dias à frente (horizonte=7).

In [ ]:
df_mimo = criar_features_multi_horizonte(df_daily, horizonte=7, max_lag=7, janela_rolling=7)
print("Shape MIMO:", df_mimo.shape)
df_mimo.head()

In [ ]:
df_mimo.to_parquet("../data/processed/mimo_features.parquet", engine="pyarrow")
print("Dados MIMO salvos!")

## Resumo

Neste notebook:

1. ✓ Corrigido data leakage do scaler (fit apenas no treino)
2. ✓ Criadas múltiplas configurações de janelas (W=3, 7, 15)
3. ✓ Preparados dados para abordagem MIMO
4. ✓ Salvos dados de treino e teste separadamente

**Próximo passo**: Executar `03_modeling.ipynb` para treinar os modelos com essas configurações.